In [1]:
from langchain_core.documents import Document

# 创建一些示例文档
documents = [
    Document(
        page_content="RAG全称为Retrieval-Augmented Generation，它通过从外部知识库中检索相关信息来增强大语言模型的回答能力。",
        metadata={"source": "doc1"}
    ),
    Document(
        page_content="向量数据库（Vector Store）是RAG系统的核心组件，用于存储文本的向量表示（Embeddings），并能高效地执行语义相似度搜索。",
        metadata={"source": "doc2"}
    ),
    Document(
        page_content="LangChain是一个用于构建基于LLM的应用的开源框架，它简化了诸如文档加载、文本拆分、与向量数据库交互等复杂流程。",
        metadata={"source": "doc3"}
    ),
    Document(
        page_content="构建RAG流程的第一步通常是加载数据，然后使用Text Splitter将其分割成适合模型处理的小块。",
        metadata={"source": "doc4"}
    )
]

print("文档加载完成！")

文档加载完成！


In [2]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

# 初始化文本拆分器
# chunk_size: 每个文本块的最大长度（字符数）
# chunk_overlap: 两个相邻块之间的重叠字符数，有助于保持上下文连续性
text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)

# 拆分文档
split_docs = text_splitter.split_documents(documents)

print(f"原始文档数量: {len(documents)}")
print(f"拆分后文档块数量: {len(split_docs)}")
print("\n--- 拆分后的第一个文档块 ---")
print(split_docs[0])

原始文档数量: 4
拆分后文档块数量: 4

--- 拆分后的第一个文档块 ---
page_content='RAG全称为Retrieval-Augmented Generation，它通过从外部知识库中检索相关信息来增强大语言模型的回答能力。' metadata={'source': 'doc1'}


In [9]:
from langchain_huggingface import HuggingFaceEmbeddings

# 初始化嵌入模型
# 我们选择一个在本地运行的轻量级模型
model_name = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name=model_name)

print("嵌入模型加载完成！")

/opt/anaconda3/envs/langchain-mcp/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


嵌入模型加载完成！


In [10]:
from langchain_community.vectorstores import FAISS

# 使用 FAISS 从文档块和嵌入模型创建向量数据库
# 这会在内存中完成所有操作
vectorstore = FAISS.from_documents(split_docs, embeddings)

print("向量数据库创建完成！")

向量数据库创建完成！


In [11]:
# 从向量数据库创建检索器
# .as_retriever() 是一个将任何 VectorStore 转换为 Retriever 的标准方法
retriever = vectorstore.as_retriever()

print("检索器创建完成！")

检索器创建完成！


In [12]:
# 定义一个自然语言查询
query = "什么是RAG？"

# 使用检索器执行查询
retrieved_docs = retriever.invoke(query)

# 打印检索结果
print(f"查询: '{query}'")
print("\n--- 检索到的相关文档 ---")
for i, doc in enumerate(retrieved_docs):
    print(f"文档 {i+1}:")
    print(f"  内容: {doc.page_content}")
    print(f"  来源: {doc.metadata['source']}")
    print("-" * 20)

# 再试一个查询
query_2 = "向量数据库有什么用？"
retrieved_docs_2 = retriever.invoke(query_2)
print(f"\n查询: '{query_2}'")
print("\n--- 检索到的相关文档 ---")
for i, doc in enumerate(retrieved_docs_2):
    print(f"文档 {i+1}:")
    print(f"  内容: {doc.page_content}")
    print(f"  来源: {doc.metadata['source']}")
    print("-" * 20)

查询: '什么是RAG？'

--- 检索到的相关文档 ---
文档 1:
  内容: 构建RAG流程的第一步通常是加载数据，然后使用Text Splitter将其分割成适合模型处理的小块。
  来源: doc4
--------------------
文档 2:
  内容: LangChain是一个用于构建基于LLM的应用的开源框架，它简化了诸如文档加载、文本拆分、与向量数据库交互等复杂流程。
  来源: doc3
--------------------
文档 3:
  内容: RAG全称为Retrieval-Augmented Generation，它通过从外部知识库中检索相关信息来增强大语言模型的回答能力。
  来源: doc1
--------------------
文档 4:
  内容: 向量数据库（Vector Store）是RAG系统的核心组件，用于存储文本的向量表示（Embeddings），并能高效地执行语义相似度搜索。
  来源: doc2
--------------------

查询: '向量数据库有什么用？'

--- 检索到的相关文档 ---
文档 1:
  内容: 构建RAG流程的第一步通常是加载数据，然后使用Text Splitter将其分割成适合模型处理的小块。
  来源: doc4
--------------------
文档 2:
  内容: LangChain是一个用于构建基于LLM的应用的开源框架，它简化了诸如文档加载、文本拆分、与向量数据库交互等复杂流程。
  来源: doc3
--------------------
文档 3:
  内容: 向量数据库（Vector Store）是RAG系统的核心组件，用于存储文本的向量表示（Embeddings），并能高效地执行语义相似度搜索。
  来源: doc2
--------------------
文档 4:
  内容: RAG全称为Retrieval-Augmented Generation，它通过从外部知识库中检索相关信息来增强大语言模型的回答能力。
  来源: doc1
--------------------
